# Проект e-commerce

Продакт-менеджер Василий попросил вас проанализировать совершенные покупки и ответить на следующие вопросы:

1. Сколько у нас пользователей, которые совершили покупку только один раз? 

2. Сколько заказов в месяц в среднем не доставляется по разным причинам (вывести детализацию по причинам)?

3. По каждому товару определить, в какой день недели товар чаще всего покупается.

4. Сколько у каждого из пользователей в среднем покупок в неделю (по месяцам)?

5. Используя pandas, провести когортный анализ пользователей. В период с января по декабрь выявить когорту с самым высоким retention на 3й месяц.

6. Используя python, построй RFM-сегментацию пользователей, чтобы качественно оценить свою аудиторию. В кластеризации можно выбрать следующие метрики: R - время от последней покупки пользователя до текущей даты, F - суммарное количество покупок у пользователя за всё время, M - сумма покупок за всё время. Подробно описать, как ты создавал кластеры. Для каждого RFM-сегмента построй границы метрик recency, frequency и monetary для интерпретации этих кластеров. Пример такого описания: RFM-сегмент 132 (recency=1, frequency=3, monetary=2) имеет границы метрик recency от 130 до 500 дней, frequency от 2 до 5 заказов в неделю, monetary от 1780 до 3560 рублей в неделю.

**Для решения задачи провести предварительное исследование данных и сформулировать, что должно считаться покупкой. Обосновать свой выбор можно с помощью фактов оплат, статусов заказов и других имеющихся данных.**

Файлы:

* **olist_customers_datase.csv — таблица с уникальными идентификаторами пользователей**

*customer_id* — позаказный идентификатор пользователя

*customer_unique_id* —  уникальный идентификатор пользователя  (аналог номера паспорта)

*customer_zip_code_prefix* —  почтовый индекс пользователя

*customer_city* —  город доставки пользователя

*customer_state* —  штат доставки пользователя

* **olist_orders_dataset.csv —  таблица заказов**

*order_id —  уникальный идентификатор заказа (номер чека)*

*customer_id —  позаказный идентификатор пользователя*

*order_status —  статус заказа*

*order_purchase_timestamp —  время создания заказа*

*order_approved_at —  время подтверждения оплаты заказа*

*order_delivered_carrier_date —  время передачи заказа в логистическую службу*

*order_delivered_customer_date —  время доставки заказа*

*order_estimated_delivery_date —  обещанная дата доставки*

* **olist_order_items_dataset.csv —  товарные позиции, входящие в заказы**

*order_id* —  уникальный идентификатор заказа (номер чека)

*order_item_id* —  идентификатор товара внутри одного заказа

*product_id* —  ид товара (аналог штрихкода)

*seller_id* — ид производителя товара

*shipping_limit_date* —  максимальная дата доставки продавцом для передачи заказа партнеру по логистике

*price* —  цена за единицу товара

*freight_value* —  вес товара

— Пример структуры данных можно визуализировать по order_id == 00143d0f86d6fbd9f9b38ab440ac16f5

Уникальные статусы заказов в таблице olist_orders_dataset:

created —  создан, 
approved —  подтверждён, 
invoiced —  выставлен счёт, 
processing —  в процессе сборки заказа, 
shipped —  отгружен со склада, 
delivered —  доставлен пользователю, 
unavailable —  недоступен, 
canceled —  отменён.

In [1]:
import pandas as pd
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [2]:
# считываем данные
customers = pd.read_csv('C:/pet/pet-projects/e-commerce/olist_customers_dataset.csv')
orders = pd.read_csv('C:/pet/pet-projects/e-commerce/olist_orders_dataset.csv')
items = pd.read_csv('C:/pet/pet-projects/e-commerce//olist_order_items_dataset.csv')

In [3]:
# проверяем на дубликаты
customers.shape[0] == customers.customer_id.nunique()

True

In [4]:
orders.shape[0] == orders.order_id.nunique()

True

In [5]:
# объединяем датафреймы для дальнейшего анализа
data = orders.merge(customers, on = 'customer_id')

In [6]:
data.dtypes

order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
customer_unique_id               object
customer_zip_code_prefix          int64
customer_city                    object
customer_state                   object
dtype: object

In [7]:
# меняем тип данных на datetime у колонок с датой
data[['order_purchase_timestamp', 
      'order_approved_at', 
      'order_delivered_carrier_date', 
      'order_delivered_customer_date', 
      'order_estimated_delivery_date']] = data[['order_purchase_timestamp', 
                                                'order_approved_at', 
                                                'order_delivered_carrier_date', 
                                                'order_delivered_customer_date', 
                                                'order_estimated_delivery_date']].apply(pd.to_datetime)

In [8]:
# будем считать покупкой только заказы со статусом "delivered", так как во всех остальных случаях заказы могут быть отменены
d_data = data[data.order_status == "delivered"]

In [9]:
# 1. Cколько у нас пользователей, которые совершили покупку только один раз?
(d_data.customer_unique_id.value_counts() == 1).sum()

90557

In [10]:
# будем считать недоставленными заказы со статусом "canceled" или "unavailable"
not_delivered_1 = data[data.order_status == "canceled"]
not_delivered_2 = data[data.order_status == "unavailable"]

In [11]:
# считаем для каждой группы среднее числов заказов в месяц
x = not_delivered_1.groupby(not_delivered_1.order_purchase_timestamp.dt.month, as_index=False) \
             .agg({'order_id': 'count'}) \
             .mean() \
             .round(0) 
y = not_delivered_2.groupby(not_delivered_2.order_purchase_timestamp.dt.month, as_index=False) \
             .agg({'order_id': 'count'}) \
             .mean() \
             .round(0)

In [12]:
# 2. Cколько заказов в месяц в среднем не доставляется по разным причинам?
print('Среднее число недоставленных заказов в месяц:', int((x+y)['order_id']))
print('Среднее число отмененных заказов за месяц:', int(x))
print('Среднее число недоступных заказов за месяц:', int(y))

Среднее число недоставленных заказов в месяц: 103
Среднее число отмененных заказов за месяц: 52
Среднее число недоступных заказов за месяц: 51


In [13]:
# проверяем на дубликаты последний датафрейм
items.duplicated().sum()

0

In [14]:
# объединяем датафреймы для анализа
order_items = items.merge(orders, on='order_id')

In [15]:
order_items[['order_purchase_timestamp', 
             'order_approved_at', 
             'order_delivered_carrier_date', 
             'order_delivered_customer_date', 
             'order_estimated_delivery_date']] = order_items[['order_purchase_timestamp', 
                                                              'order_approved_at', 
                                                              'order_delivered_carrier_date', 
                                                              'order_delivered_customer_date', 
                                                              'order_estimated_delivery_date']].apply(pd.to_datetime)

In [16]:
# т.к покупкой мы считаем только доставленные заказы, фильтруем по их по статусу для дальнейшего анализа
d_order_items = order_items[order_items.order_status == "delivered"]

In [17]:
# для начала посчитаем число покупок каждого товара по дням недели
amount = d_order_items.groupby([d_order_items.order_purchase_timestamp.dt.day_name(), d_order_items.product_id]) \
                 .order_item_id.count() \
                 .reset_index() \
                 .rename(columns={'order_purchase_timestamp' : 'weekday'})

In [18]:
# 3. По каждому товару определяем, в какой день недели товар чаще всего покупается
amount.loc[amount.groupby('product_id').order_item_id.idxmax()] \
     .reset_index() \
     .drop(columns = 'index') \
     .rename(columns={'order_item_id' : 'quantity'})

,weekday,product_id,quantity
0,Sunday,00066f42aeeb9f3007548bb9d3f33c38,1
1,Tuesday,00088930e925c41fd95ebfe695fd2655,1
2,Thursday,0009406fd7479715e4bef61dd91f2462,1
3,Friday,000b8f95fcb9e0096488278317764d19,1
4,Tuesday,000d9be29b5207b54e86aa1b1ac54872,1
...,...,...,...
32211,Saturday,fff6177642830a9a94a0f2cba5e476d1,1
32212,Monday,fff81cc3158d2725c0655ab9ba0f712c,1
32213,Friday,fff9553ac224cec9d15d49f5a263411f,1
32214,Tuesday,fffdb2d0ec8d6a61f0a0a0db3f25b441,2


In [19]:
# для подсчета среднего числа покупок у пользователей в неделю, группируем по месяцу и пользователю
# опять берем заказы только со статусом "delivered"
purchases = d_data.groupby(['customer_unique_id', pd.Grouper(key='order_purchase_timestamp', freq='M')]) \
                  .order_id.count() \
                  .reset_index()

In [20]:
# создаем новые колонки
purchases['weeksinmonth'] = purchases.order_purchase_timestamp.dt.daysinmonth / 7
purchases['weekly_purchases'] = purchases.order_id/purchases.weeksinmonth

In [21]:
# 4. Сколько у каждого из пользователей в среднем покупок в неделю (по месяцам)
purchases.groupby('customer_unique_id').weekly_purchases.mean()

customer_unique_id
0000366f3b9a7992bf8c76cfdf3221e2    0.225806
0000b849f77a49e4a4ce2b2a4ca5be3f    0.225806
0000f46a3911fa3c0805444483337064    0.225806
0000f6ccb0745a6a4b88665a16c9f078    0.225806
0004aac84e0df4da2b147fca70cf8255    0.233333
                                      ...   
fffcf5a5ff07b0908bd4e2dbc735a684    0.233333
fffea47cd6d3cc0a88bd621562a9d061    0.225806
ffff371b4d645b6ecea244b27531430a    0.250000
ffff5962728ec6157033ef9805bacc48    0.225806
ffffd2657e2aad2907e67c3e9daecbeb    0.225806
Name: weekly_purchases, Length: 93358, dtype: float64

In [22]:
# Начинаем когортный анализ
# Сначала находим дату первой покупки для каждого пользователя
# Мы работаем только с ордерами со статусом "delivered"
first_purchase = d_data.groupby('customer_unique_id').order_purchase_timestamp.min().reset_index()

In [23]:
# После этого разбиваем эти даты по месяцам, это и будут когорты
first_purchase['cohort'] = first_purchase.order_purchase_timestamp.dt.to_period('M')
first_purchase = first_purchase.rename(columns={'order_purchase_timestamp' : 'first_purchase'})

In [24]:
# Добавляем новые колонки с датой первой покупки и когортой к основному датафрейму с помощью джойна
d_data = d_data.merge(first_purchase, on='customer_unique_id')

In [25]:
# Группируем по когорте и месяцу покупки и считаем число уникальных покупателей для каждой группы
cohort_data = d_data.groupby([d_data.cohort, d_data.order_purchase_timestamp.dt.to_period('M')]) \
                    .customer_unique_id.nunique() \
                    .reset_index()

In [26]:
# создаем новые колонки, чтобы определить число с месяцев с образования когорты и retention
cohort_data['month_number'] = cohort_data.order_purchase_timestamp.astype(int) - cohort_data.cohort.astype(int)
cohort_data['retention'] = cohort_data['customer_unique_id'] / cohort_data.groupby('cohort')['customer_unique_id'].transform('first')

In [27]:
# 5. Найдем когорту с самым высоким retention на 3й месяц
cohort_data[cohort_data.month_number == 3].sort_values(by='retention', ascending=False).head(1)

,cohort,order_purchase_timestamp,customer_unique_id,month_number,retention
102,2017-06,2017-09,13,3,0.004281


In [28]:
# Чтобы начать делать сегментацию нам необходимо объединить все таблицы
# Берем уже готовую таблицу с клиентами и доставленными заказами, приклеиваем информацию по товарам
all_data = d_data.merge(items, on='order_id')

In [29]:
# Находим стоимость заказов
order_price = all_data.groupby(['order_id', 'customer_unique_id', 'order_purchase_timestamp']).price.sum().reset_index()

In [30]:
# Т.к данные старые, то будем считать, что "сейчас" - это последняя дата покупки + 1 день
NOW = order_price.order_purchase_timestamp.max() + pd.Timedelta(days=1)
NOW

Timestamp('2018-08-30 15:00:37')

In [31]:
# создаем колонку с разницей дней между "сейчас" и датой покупки
order_price['days'] = (NOW - order_price['order_purchase_timestamp']).dt.days

In [32]:
# создаем новый датафрейм: группируем по пользователям, после этого считаем Recency, Frequency и Monetary
# Recency - число дней с последней покупки, Frequency - число покупок за все время, Monetary - количество денег за все время
rfm = order_price.groupby('customer_unique_id', as_index = False) \
                .agg({'days': 'min', 'order_id': 'count', 'price': 'sum'}) \
                .rename(columns={'days':'Recency', 'order_id': 'Frequency', 'price': 'Monetary'})

In [33]:
# Делим наши данные на сегменты 
# Т.к у нас очень малое число повторных покупок (>1), то для Frequency берем 2 группы
# Для остальных характеристик по 4 группы
quintiles_1 = rfm[['Recency']].quantile([.25, .5, .75]).to_dict()
quintiles_3 = rfm[['Monetary']].quantile([.25, .5, .75]).to_dict()

In [34]:
def r_score(x):
    if x <= quintiles_1['Recency'][.25]:
        return 4
    elif x <= quintiles_1['Recency'][.5]:
        return 3
    elif x <= quintiles_1['Recency'][.75]:
        return 2
    else:
        return 1

def f_score(x):
    if x <= 1:
        return 1
    else:
        return 4  

def m_score(x):
    if x <= quintiles_3['Monetary'][.25]:
        return 1
    elif x <= quintiles_3['Monetary'][.5]:
        return 2
    elif x <= quintiles_3['Monetary'][.75]:
        return 3
    else:
        return 4

In [35]:
# Применяем функции, чтобы присвоить необходимый скор и делаем новую колонку
# Т.к во Frequency всего 2 группы то F либо 1 (меньше одной покупки), либо 4 (больше одной покупки)
rfm['R'] = rfm['Recency'].apply(r_score)
rfm['F'] = rfm['Frequency'].apply(f_score)
rfm['M'] = rfm['Monetary'].apply(m_score)
rfm['RFM Score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)

In [36]:
# Делаем словарь, в котором обозначаем границы каждого из сегментов
# За образец возьмем сегменты из примера с поправкой на наши данные
# границы сегментов можно увидеть из структуры словаря и значений квантилей
segt_map = {
    r'11[1-4]': 'hibernating',
    r'14[1-2]': 'at risk',
    r'14[3-4]': 'can\'t loose',
    r'21[1-3]': 'about to sleep',
    r'2[1-4][1-4]': 'need attention',
    r'[3-4]4[3-4]': 'loyal customers',
    r'[3-4]1[3-4]': 'promising',
    r'[3-4]1[1-2]': 'new customers',
    r'[3-4]4[1-2]': 'potential loyalists'
}

# hibernating - последняя покупка была очень давно, малое число покупок, могут быть уже не активны
# at risk - последняя покупка также была очень давно, однако было больше одной покупки
# can't loose - последняя покупка была очень давно, однако было больше одной покупки и клиент в топе по потраченным деньгам
# about to sleep - давно не было покупок
# need attention - давно не было покупок, при этом клиенты из этой группы либо часто покупали, либо в топе по потраченным деньгам
# loyal customers - наши лучшие клиенты, регулярно совершают покупки и больше всего тратят
# promising - недавно соверишили покупку, тратят выше среднего
# new customers - недавно совершили покупку на небольшую сумму
# potential loyalists - недавно совершили несколько покупок, но на не очень большую сумму


rfm['Segment'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)
rfm['Segment'] = rfm['Segment'].replace(segt_map, regex=True)
rfm.Segment.value_counts()

new customers          22938
hibernating            22732
promising              22360
about to sleep         17125
need attention          6075
loyal customers         1316
can't loose              463
potential loyalists      233
at risk                  116
Name: Segment, dtype: int64